# "자비서" 호출어 openWakeWord 분류기 학습 6회차 → `jarviseo-r6.onnx`

**6회차는 5회차(`jarviseo_train_r5.ipynb`)에서 데이터만 바꿨다.** `jarviseo-r6.zip` = 5회차 데이터 + 평가용과 다른 방송을 같은 노트북 마이크로 60분 녹음한 것(train/background, 2초 조각)을 negative 로 추가 (negative 학습 4150·검증 471). 학습 설정(학습률 warmup+코사인, EMA 0.999, 50000 스텝, 배치, 실제 녹음 10, negative 가중치, 모델 고르는 기준)은 5회차와 같다. positive 특징은 5회차와 같은 클립·난수열이라 `features/run4` 의 positive 두 묶음을 복사해 쓰고, negative 두 묶음만 새로 계산한다. 스트리밍 확인 표에 negative_test 의 label(background·similar·speech)별 FAR 을 더했다.

**5회차는 4회차(`jarviseo_train_r4.ipynb`)에서 학습만 안정화했다.** 데이터·특징은 4회차 그대로(`jarviseo-r4.zip`, `features/run4` 재사용). 바꾼 것: 학습률 워밍업 1000 스텝 + 코사인 감소(1e-4 → 1e-6), 가중치 이동 평균(EMA, decay 0.999). 검증·스냅샷·ONNX·스트리밍 확인은 모두 EMA 가중치로 한다. (4회차는 학습 체크포인트마다 시간당 오탐이 크게 출렁였다.)

**4회차는 3회차(`jarviseo_train_r3.ipynb`)에서 데이터만 바꿨다** (`jarviseo-r4.zip` = 3회차 + hard negative 600, negative 학습 2530·검증 291). 증강, 모델 구조, 스텝 수, lr, negative 가중치, 배경 특징, positive 배치의 실제 녹음 10개, 모델 고르는 기준(10000 스텝부터, t*), 확인 표는 3회차와 똑같다.

아래 1~4 는 3회차에서 2회차 대비 바꾼 내용이며 4회차에도 그대로 적용된다.

1. **데이터와 이름**: `jarviseo-r4.zip`, `features/run4`(4회차와 같음), 결과 `jarviseo-r5.onnx`·`jarviseo-r5-train.json`.
   `split.csv` 의 `text` 열로 "자비서" 클립의 말투를 나눈다(끝의 `!`·`?` 는 떼고 비교).
   끊음 = "자, 비서", "자... 비서", "자비, 서", "자, 비, 서" / 늘임 = "자~비~서", "자아비서", "자비이서", "자아비이서어" / 나머지는 정상.
2. **positive 배치에 실제 녹음을 더 넣기**: 한 스텝 positive 50개 중 `BATCH_POS_MIC`=10개는 positive_train 의 실제 녹음(source == "mic")에서,
   나머지 40개는 positive_train 전체에서 뽑는다.
3. **가장 좋은 모델 고르는 기준**: 후보는 `STEPS` 의 20%(10000 스텝)부터의 검증 시점만 쓴다.
   스텝마다 시간당 오탐 ≤ 0.5 를 맞추는 가장 낮은 임계값 t* 를 `TH_GRID` 에서 찾고, t* 에서의 recall 로 비교한다
   (같으면 시간당 오탐이 낮은 쪽, 그다음은 늦은 스텝). 어떤 후보도 0.95 에서 맞추지 못하면 0.95 에서 시간당 오탐이 가장 낮은 스텝을 고르고 경고한다.
   t* 는 참고값이고 실제 임계값은 학습 뒤 eval 로 정한다.
4. **스트리밍 확인 표**: 2회차 표(임계값 0.1~0.9, 전체/합성/실제)에 더해 positive_test 를 말투(정상·끊음·늘임)별로 나눈 FRR 표를 임계값 0.5 와 t* 에서 낸다.
   1·2회차와 나란히 놓는 비교표는 뺐다(test 구성이 다르다).

openWakeWord 는 세 단계로 동작합니다.

1. `melspectrogram.onnx` (고정)
2. `embedding_model.onnx` = Google speech_embedding (고정)
3. **작은 분류기** ← 이 노트북에서 학습하는 부분. 입력은 임베딩 16개 × 96 = `(16, 96)` float32, 출력은 0~1 점수 1개.

- 공식 학습 코드(`openwakeword/train.py`, `openwakeword/data.py`)는 쓰지 않습니다. 증강과 학습 반복문은 numpy·scipy·torch 로 직접 구현했습니다.
- **런타임**: 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`. GPU 가 없으면 자동으로 CPU 로 돕니다(느리지만 동작).
- **학습 데이터**: `MyDrive/jarviseo/jarviseo-r4.zip`. 안에 `jarviseo/positive_train`(3514), `positive_test`(384), `negative_train`(2530), `negative_test`(291), `split.csv` 가 있습니다.
  **폴더 구성을 그대로 쓰고, 다시 나누거나 자르거나 거르지 않습니다.** 합성/실제 구분은 `split.csv` 의 `out`·`source` 열로 합니다.
- **필요 자원**: 디스크 여유 약 20GB(배경 특징 파일 17.3GB), RAM 약 8GB(배경 특징 4.6GB를 메모리에 올림).
- **평가용 녹음(eval)은 이 노트북에서 쓰지 않습니다.** zip 에 `eval` 폴더가 있으면 멈춥니다. 최종 평가는 학습 뒤 `evaluate.py --model` 로 따로 합니다.
- 각 코드 셀 맨 위의 `%%time` 이 셀이 끝날 때 걸린 시간(Wall time)을 출력합니다.
- 위에서부터 순서대로 실행하세요. 증강 특징(.npy)은 Drive 에 저장되므로, 다시 돌릴 때는 증강 단계를 건너뜁니다.

## 0. 설정값
경로, 증강, 모델 크기, 학습 스텝 수 등 바꿀 수 있는 값을 모두 여기 모았습니다. 이 셀만 고치면 됩니다.
증강 관련 값을 바꿨다면 `FORCE_RECOMPUTE_FEATURES = True` 로 두거나 Drive 의 `features/run4/*.npy` 를 지우세요.

In [ ]:
%%time
# ===== 경로 =====
RUN_TAG = "r6"                                                 # 6회차 (r5 + 방송 녹음 negative)
ZIP_PATH = "/content/drive/MyDrive/jarviseo/jarviseo-r6.zip"  # 학습 데이터 zip (폴더 4개 + split.csv)
OUT_DIR = "/content/drive/MyDrive/jarviseo"                    # 결과(onnx, json)를 저장할 Drive 폴더
FEAT_DIR = f"{OUT_DIR}/features/run6"                          # 6회차 특징 캐시
REUSE_POS_FEAT_DIR = f"{OUT_DIR}/features/run4"                # positive 두 묶음은 r4·r5 와 같은 클립·난수열 → 여기서 복사 (None 이면 새로 계산)
DATA_DIR = "/content/data"                                     # zip 을 풀 위치
OWW_DATA = "/content/oww_data"                                 # 공개 자료(배경 특징, RIR, AudioSet) 위치
MODEL_NAME = "jarviseo"

# ===== 무작위 시드 =====
SEED = 0

# ===== zip 의 예상 개수 (wav 기준) =====
EXPECTED_COUNTS = {"positive_train": 3514, "positive_test": 384, "negative_train": 4150, "negative_test": 471}
EXPECTED_TEST_BY_SOURCE = {"positive_test": {"tts": 375, "mic": 9}, "negative_test": {"tts": 285, "mic": 186}}

# ===== "자비서" 말투 (split.csv 의 text, 끝의 !·? 는 떼고 비교) =====
STYLE_CUT = ["자, 비서", "자... 비서", "자비, 서", "자, 비, 서"]          # 끊음
STYLE_STRETCH = ["자~비~서", "자아비서", "자비이서", "자아비이서어"]     # 늘임 (나머지는 정상)

# ===== 증강 (클립 하나 → 2초 창 하나) — 2회차와 같음 =====
SR = 16000
WINDOW_SAMPLES = 32000                 # 2초 창
END_PAD_MAX_S = 0.2                    # 클립이 끝난 뒤 0~0.2초 여유
GAIN_DB = (-6.0, 6.0)                  # 음량 변화 (항상)
BG_NOISE_P, BG_SNR_DB = 0.75, (-5.0, 15.0)        # AudioSet 배경 소음
RIR_P = 0.5                                        # 잔향 컨볼루션
COLOR_NOISE_P, COLOR_SNR_DB = 0.25, (10.0, 30.0)  # 색잡음
AUG_ROUNDS = {"positive_train": 3, "negative_train": 2, "positive_test": 1, "negative_test": 1}
FORCE_RECOMPUTE_FEATURES = False       # True 면 Drive 에 특징 파일이 있어도 다시 계산
EMBED_BATCH = 128                      # embed_clips batch_size
EMBED_NCPU = 4                         # embed_clips ncpu

# ===== 배경 특징 / 오탐 검증 — 2회차와 같음 =====
BG_ROWS = 1_500_000                    # RAM 에 올릴 배경 특징 행 수 (float16 약 4.6GB)
BG_BLOCK = 10_000                      # 한 번에 읽는 연속 행 수
VAL_HOURS = 10.7                       # 오탐 검증 특징 길이(시간)

# ===== 모델 (openWakeWord dnn 구조) — 2회차와 같음 =====
LAYER_DIM = 32                         # L
N_BLOCKS = 1                           # [Linear → LayerNorm → ReLU] 반복 수

# ===== 학습 — 2회차와 같음 (BATCH_POS_MIC 만 새로 생김) =====
STEPS = 50_000
LR = 1e-4
BATCH_POS, BATCH_NEG, BATCH_BG = 50, 50, 1024
BATCH_POS_MIC = 10                     # positive 50 중 실제 녹음(mic)에서 뽑는 개수. 나머지 40 은 positive_train 전체에서
MAX_NEG_WEIGHT = 1500                  # negative 가중치: 1 → MAX_NEG_WEIGHT 선형 증가
LR_WARMUP_STEPS = 1000                 # (5회차) 0 → LR 선형 워밍업 스텝
LR_MIN = 1e-6                          # (5회차) 워밍업 뒤 STEPS 까지 코사인으로 내려갈 최소 학습률
EMA_DECAY = 0.999                      # (5회차) 가중치 이동 평균. 검증·내보내기는 EMA 가중치로
EVAL_EVERY = 500
SCORE_THRESHOLD = 0.5                  # 기본 임계값 (표의 기준)
FP_COOLDOWN_WINDOWS = 25               # 오탐 1회 후 다음 25창(2초)은 세지 않음

# ===== 가장 좋은 모델 고르기 (3회차에서 바뀜) =====
TARGET_FP_PER_HOUR = 0.5               # 시간당 오탐 목표
SELECT_FROM_FRAC = 0.2                 # 후보는 STEPS 의 이 비율(10000 스텝)부터의 검증 시점만
TH_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]   # 검증 때 임계값마다 recall·negative 오탐률·시간당 오탐을 기록

# ===== 내보내기 / 확인 =====
ONNX_OPSET = 13
ONNX_PATH = f"{OUT_DIR}/{MODEL_NAME}-{RUN_TAG}.onnx"           # jarviseo-r6.onnx
JSON_PATH = f"{OUT_DIR}/{MODEL_NAME}-{RUN_TAG}-train.json"     # jarviseo-r6-train.json
CHECK_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# JSON 에 남길 설정값 (위의 대문자 변수들)
CONFIG = {k: v for k, v in list(globals().items())
          if k.isupper() and k != "CONFIG" and isinstance(v, (int, float, str, bool, tuple, list, dict))}
print(f"설정값 {len(CONFIG)}개")

## 1. 설치
- `openwakeword==0.6.0` 은 `--no-deps` 로 설치합니다. 리눅스 의존성 `tflite-runtime` 은 파이썬 3.12 이상용 wheel 이 없기 때문입니다. 추론은 onnxruntime 으로 합니다.
- `onnx`, `onnxscript` 는 `torch.onnx.export` 와 opset 확인에 씁니다.
- 마지막에 특징 모델(`melspectrogram.onnx`, `embedding_model.onnx`)을 받습니다.

In [ ]:
%%time
!pip install openwakeword==0.6.0 --no-deps
!pip install onnxruntime tqdm scikit-learn soundfile pyarrow huggingface_hub
!pip install onnx onnxscript

import os
import openwakeword
import openwakeword.utils

openwakeword.utils.download_models(["hey_jarvis"])
OWW_MODEL_DIR = os.path.join(os.path.dirname(openwakeword.__file__), "resources", "models")
for f in ["melspectrogram.onnx", "embedding_model.onnx"]:
    p = os.path.join(OWW_MODEL_DIR, f)
    assert os.path.exists(p), f"특징 모델이 없습니다: {p}"
    print(f"{f}: {os.path.getsize(p) / 1e6:.2f} MB")

## 2. 라이브러리 불러오기, 시드 고정, 장치 확인
- 학습 장치: CUDA 가 있으면 GPU, 없으면 CPU.
- 특징 추출(`AudioFeatures`) 장치: GPU 런타임이면 `"gpu"`. 단 설치된 onnxruntime 에 CUDA 실행기가 없으면 오류가 나므로 그때는 `"cpu"` 로 바꿉니다.
- 공통 도우미 함수(RMS, 리샘플)도 여기서 정의합니다.

In [ ]:
%%time
import os, io, gc, json, math, glob, shutil, random, zipfile, datetime, importlib.metadata
import numpy as np
import pandas as pd
import scipy.signal
import soundfile as sf
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import onnxruntime as ort
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, snapshot_download
from openwakeword.utils import AudioFeatures
from openwakeword.model import Model as OWWModel


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def rms(x):
    return float(np.sqrt(np.mean(np.square(x, dtype=np.float64)) + 1e-12))


def resample_to(x, sr_in, sr_out=16000):
    # 정수비 다상 리샘플 (예: 44100 → 16000 은 160/441)
    sr_in, sr_out = int(sr_in), int(sr_out)
    if sr_in == sr_out:
        return x.astype(np.float32)
    g = math.gcd(sr_in, sr_out)
    return scipy.signal.resample_poly(x, sr_out // g, sr_in // g).astype(np.float32)


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feat_device = "gpu" if torch.cuda.is_available() else "cpu"
if feat_device == "gpu" and "CUDAExecutionProvider" not in ort.get_available_providers():
    print("설치된 onnxruntime 에 CUDAExecutionProvider 가 없어 특징 추출은 CPU 로 합니다. (학습은 GPU)")
    feat_device = "cpu"

print("torch", torch.__version__, "| onnxruntime", ort.__version__, "| numpy", np.__version__,
      "| openwakeword", importlib.metadata.version("openwakeword"))
print("학습 장치:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU 없음")
print("특징 추출 장치:", feat_device, "| ORT providers:", ort.get_available_providers())
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM {ram_gb:.1f} GB | /content 디스크 여유 {shutil.disk_usage('/content').free / 1e9:.1f} GB")

## 3. 데이터
### 3-1. Google Drive 마운트, 학습 zip 풀기, 말투 나누기
- `ZIP_PATH` 를 `/content/data/r4` 에 풉니다. 이미 풀려 있고 개수가 맞으면 건너뜁니다.
- **폴더 구성을 그대로 씁니다.** 다시 나누거나 자르거나 거르지 않습니다. 폴더 안 wav 개수가 예상과 다르면 멈춥니다.
- 합성/실제 구분은 `split.csv` 의 `out`(폴더/파일명)과 `source` 열로 합니다. 확인 표의 개수는 wav 기준입니다.
- "자비서" 클립의 말투는 `split.csv` 의 `text` 열로 나눕니다(끝의 `!`·`?` 는 떼고 비교): 끊음 / 늘임 / 정상.

In [ ]:
%%time
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

splits = ["positive_train", "positive_test", "negative_train", "negative_test"]


def find_data_root(base):
    if not os.path.isdir(base):
        return None
    for root, dirs, _ in os.walk(base):
        if all(s in dirs for s in splits):
            return root
    return None


def list_wavs(root, split):
    return sorted(glob.glob(os.path.join(root, split, "*.wav")))


extract_to = os.path.join(DATA_DIR, RUN_TAG)
data_root = find_data_root(extract_to)
if data_root and all(len(list_wavs(data_root, s)) == n for s, n in EXPECTED_COUNTS.items()):
    print("이미 풀려 있어 건너뜁니다:", data_root)
else:
    assert os.path.exists(ZIP_PATH), f"zip 파일이 없습니다: {ZIP_PATH}"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        members = [m for m in zf.infolist() if not m.filename.startswith("__MACOSX")]
        assert not any("/eval/" in "/" + m.filename for m in members), "zip 에 eval 폴더가 있습니다. 평가용 녹음은 쓰지 않습니다"
        for m in tqdm(members, desc="압축 풀기"):
            zf.extract(m, extract_to)
    data_root = find_data_root(extract_to)
assert data_root, "positive_train / positive_test / negative_train / negative_test 폴더를 찾지 못했습니다"
print("학습 데이터 폴더:", data_root)

# 폴더 구성을 그대로 씁니다 (다시 나누거나 거르지 않음)
wav_files = {s: list_wavs(data_root, s) for s in splits}
for s in splits:
    n, e = len(wav_files[s]), EXPECTED_COUNTS[s]
    print(f"{s:15s} {n:5d}개 (예상 {e})")
    assert n == e, f"{s} 개수가 예상과 다릅니다: {n} != {e}"

# split.csv 의 out(폴더/파일명) → source
split_df = pd.read_csv(os.path.join(data_root, "split.csv"), dtype=str, keep_default_na=False)
print("\nsplit.csv:", split_df.shape, list(split_df.columns))
for col in ["out", "source", "text"]:
    assert col in split_df.columns, f"split.csv 에 {col} 열이 없습니다"
out_keys = split_df.out.str.replace("\\", "/", regex=False)
source_of = dict(zip(out_keys, split_df.source))
text_of = dict(zip(out_keys, split_df.text))
label_of = dict(zip(out_keys, split_df.label)) if "label" in split_df.columns else {}


def style_of(text):
    # 끝의 !·? 는 떼고 비교
    t = str(text).strip().rstrip("!?").strip()
    if t in STYLE_CUT:
        return "끊음"
    if t in STYLE_STRETCH:
        return "늘임"
    return "정상"


wav_sources, wav_texts, wav_styles, wav_labels = {}, {}, {}, {}
for s in splits:
    keys = [f"{s}/{os.path.basename(p)}" for p in wav_files[s]]
    missing = [k for k in keys if k not in source_of]
    assert not missing, f"{s}: split.csv 에 없는 wav {len(missing)}개 (예: {missing[:3]})"
    wav_sources[s] = np.array([source_of[k] for k in keys])
    wav_texts[s] = np.array([text_of[k] for k in keys])
    wav_labels[s] = np.array([label_of.get(k, "?") for k in keys])
    wav_styles[s] = np.array([style_of(t) for t in wav_texts[s]])

src_table = pd.DataFrame({s: pd.Series(wav_sources[s]).value_counts() for s in splits}).T.fillna(0).astype(int)
src_table["All"] = src_table.sum(axis=1)
print("\nwav 기준 개수 (source 별):")
print(src_table)
for s, exp in EXPECTED_TEST_BY_SOURCE.items():
    got = {k: int((wav_sources[s] == k).sum()) for k in exp}
    print(f"{s}: {got} (예상 {exp})" + ("" if got == exp else "   ※ 예상과 다름"))

# "자비서" 말투 (positive 만)
style_table = pd.DataFrame({s: pd.Series(wav_styles[s]).value_counts()
                            for s in ["positive_train", "positive_test"]}).T.reindex(columns=["정상", "끊음", "늘임"]).fillna(0).astype(int)
print("\n\"자비서\" 말투별 개수 (split.csv 의 text 기준):")
print(style_table)
for s in ["positive_train", "positive_test"]:
    print(f"{s} 말투×source:", pd.crosstab(wav_styles[s], wav_sources[s]).to_dict())
pos_texts = pd.Series(np.concatenate([wav_texts["positive_train"], wav_texts["positive_test"]]))
print("positive text 종류(상위 20):", pos_texts.value_counts().head(20).to_dict())

# 형식 확인
info_rows = []
for s in splits:
    for p in tqdm(wav_files[s], desc=f"{s} 형식 확인", leave=False):
        i = sf.info(p)
        info_rows.append((s, i.samplerate, i.channels, i.subtype, i.frames / i.samplerate))
info_df = pd.DataFrame(info_rows, columns=["split", "sr", "ch", "subtype", "sec"])
print("\n샘플레이트:", sorted(info_df.sr.unique()), "| 채널:", sorted(info_df.ch.unique()),
      "| 형식:", sorted(info_df.subtype.unique()))
print("길이(초):")
print(info_df.groupby("split").sec.describe()[["mean", "min", "max"]].round(2))
n_long = int((info_df.sec > WINDOW_SAMPLES / SR + 1e-6).sum())
print(f"2초 넘는 클립: {n_long}개" + ("" if n_long == 0 else " (증강 때 끝 2초만 씀)"))

print("\nnegative label 별 개수:", {s: pd.Series(wav_labels[s]).value_counts().to_dict() for s in ["negative_train", "negative_test"]})
print("negative_test label×source:", pd.crosstab(wav_labels["negative_test"], wav_sources["negative_test"]).to_dict())

feature_paths = {s: os.path.join(FEAT_DIR, f"{s}.npy") for s in splits}

# positive 특징 재사용 (r4·r5 와 같은 클립·같은 난수열). 행 수가 클립 수 × 증강 횟수와 맞는지 확인하고 복사
if REUSE_POS_FEAT_DIR and not FORCE_RECOMPUTE_FEATURES:
    for s in ["positive_train", "positive_test"]:
        src_p, dst_p = os.path.join(REUSE_POS_FEAT_DIR, f"{s}.npy"), feature_paths[s]
        if os.path.exists(dst_p) or not os.path.exists(src_p):
            continue
        rows = np.load(src_p, mmap_mode="r").shape[0]
        expect = len(wav_files[s]) * AUG_ROUNDS[s]
        if rows == expect:
            shutil.copy(src_p, dst_p)
            print(f"{s}: {REUSE_POS_FEAT_DIR} 의 특징 복사 ({rows}행)")
        else:
            print(f"※ {s}: 재사용 특징 행 수 {rows} ≠ 예상 {expect} → 새로 계산")
need_aug = FORCE_RECOMPUTE_FEATURES or not all(os.path.exists(p) for p in feature_paths.values())
print("\n특징 캐시 폴더:", FEAT_DIR, "| 증강·특징 계산 필요:", need_aug, "(False 면 RIR·AudioSet 다운로드도 건너뜀)")

### 3-2. 오탐 검증 특징 (validation_set_features.npy)
185MB, float32, `(481345, 96)`. 80ms 마다 이어지는 프레임(약 10.7시간)입니다. 나중에 16프레임씩 1프레임 간격으로 밀어 창을 만듭니다.

In [ ]:
%%time
os.makedirs(OWW_DATA, exist_ok=True)
val_path = hf_hub_download(repo_id="davidscripka/openwakeword_features", repo_type="dataset",
                           filename="validation_set_features.npy", local_dir=OWW_DATA)
val_feats = np.load(val_path).astype(np.float32)
n_val_windows = val_feats.shape[0] - 16 + 1
print("오탐 검증 특징:", val_feats.shape, val_feats.dtype)
print(f"길이 {val_feats.shape[0] * 0.08 / 3600:.2f}시간 (계산은 VAL_HOURS={VAL_HOURS}) | 16프레임 창 {n_val_windows:,}개")

### 3-3. 배경 특징 (ACAV100M 2000시간, 17.3GB) 받기 → 일부를 RAM 에 올리기
float16, `(5625000, 16, 96)`. 이미 16프레임 창으로 잘린 일반 음성·소리 특징입니다.
`mmap_mode="r"` 로 열고, 행 단위 무작위 접근은 느리므로 **무작위 위치의 연속 10000행 블록**을 읽어 `BG_ROWS` 행을 RAM 에 올립니다.
다운로드에 보통 5~20분 걸립니다.

In [ ]:
%%time
BG_FILE = "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
bg_local = os.path.join(OWW_DATA, BG_FILE)
if not os.path.exists(bg_local):
    free = shutil.disk_usage(OWW_DATA).free
    print(f"디스크 여유 {free / 1e9:.1f} GB (필요 17.3 GB)")
    if free < 17.3e9 * 1.02:
        raise RuntimeError("디스크 공간이 부족합니다. 런타임을 초기화하거나 다른 파일을 지우세요.")
bg_path = hf_hub_download(repo_id="davidscripka/openwakeword_features", repo_type="dataset",
                          filename=BG_FILE, local_dir=OWW_DATA)

bg_mm = np.load(bg_path, mmap_mode="r")
print("배경 특징 파일:", bg_mm.shape, bg_mm.dtype)
assert bg_mm.shape[1:] == (16, 96)

n_blocks_total = bg_mm.shape[0] // BG_BLOCK
n_blocks = min(math.ceil(BG_ROWS / BG_BLOCK), n_blocks_total)
block_ids = np.sort(np.random.default_rng([SEED, 1]).choice(n_blocks_total, size=n_blocks, replace=False))
bg_feats = np.empty((n_blocks * BG_BLOCK, 16, 96), dtype=np.float16)
for j, b in enumerate(tqdm(block_ids, desc="배경 특징 RAM 적재 (블록)")):
    bg_feats[j * BG_BLOCK:(j + 1) * BG_BLOCK] = bg_mm[b * BG_BLOCK:(b + 1) * BG_BLOCK]
bg_feats = bg_feats[:BG_ROWS]
del bg_mm
gc.collect()
print(f"RAM 의 배경 특징: {bg_feats.shape} {bg_feats.dtype}, {bg_feats.nbytes / 1e9:.2f} GB")

### 3-4. 잔향(RIR): MIT environmental impulse responses (16kHz)
각 RIR 은 최댓값(직접음) 위치부터 쓰도록 앞을 잘라, 컨볼루션 뒤에도 말소리 위치가 밀리지 않게 합니다.
특징 파일이 이미 다 있으면 건너뜁니다.

In [ ]:
%%time
rirs = []
if need_aug:
    rir_dir = snapshot_download(repo_id="davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                                allow_patterns="16khz/*", local_dir=os.path.join(OWW_DATA, "mit_rirs"))
    for p in tqdm(sorted(glob.glob(os.path.join(rir_dir, "16khz", "*.wav"))), desc="RIR 읽기"):
        x, sr = sf.read(p, dtype="float32", always_2d=True)
        x = resample_to(x.mean(axis=1), sr, SR)
        x = x[int(np.argmax(np.abs(x))):]
        energy = float(np.sqrt(np.sum(x.astype(np.float64) ** 2)))
        if energy > 0:
            rirs.append((x / energy).astype(np.float32))
    print(f"RIR {len(rirs)}개, 길이 {np.mean([len(r) for r in rirs]) / SR:.2f}초(평균)")
else:
    print("특징 파일이 모두 있어 건너뜁니다")

### 3-5. 배경 소음: AudioSet `data/bal_train/00.parquet` (688MB, 10초 클립 약 500개)
pyarrow 로 읽어 `audio` 열의 bytes 를 soundfile 로 디코딩 → 모노 → 16kHz 리샘플(`resample_poly`) → float32 목록.
디코딩이 안 되는 행은 건너뛰고 개수를 출력합니다. (완전 무음 클립도 SNR 을 맞출 수 없어 따로 세고 뺍니다.)

In [ ]:
%%time
noises = []
audioset_stats = {"rows": 0, "decoded": 0, "decode_failed": 0, "silent_skipped": 0, "skipped_cached": not need_aug}
if need_aug:
    pq_path = hf_hub_download(repo_id="agkphysics/AudioSet", repo_type="dataset",
                              filename="data/bal_train/00.parquet", local_dir=os.path.join(OWW_DATA, "audioset"))
    pf = pq.ParquetFile(pq_path)
    audioset_stats["rows"] = pf.metadata.num_rows
    with tqdm(total=pf.metadata.num_rows, desc="AudioSet 디코딩") as pbar:
        for batch in pf.iter_batches(batch_size=16, columns=["audio"]):
            for item in batch.column(0).to_pylist():
                pbar.update(1)
                try:
                    raw = item["bytes"] if isinstance(item, dict) else item
                    data, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
                    x = resample_to(data.mean(axis=1), sr, SR)
                    if x.size == 0:
                        raise ValueError("빈 오디오")
                except Exception:
                    audioset_stats["decode_failed"] += 1
                    continue
                if rms(x) < 1e-4:
                    audioset_stats["silent_skipped"] += 1
                    continue
                noises.append(x)
    audioset_stats["decoded"] = len(noises)
    del pf
    gc.collect()
    print(f"배경 소음 {len(noises)}개 사용 | 디코딩 실패로 건너뜀 {audioset_stats['decode_failed']}개 "
          f"| 무음이라 뺀 것 {audioset_stats['silent_skipped']}개 | 전체 {audioset_stats['rows']}행")
    print(f"총 길이 {sum(len(n) for n in noises) / SR / 60:.1f}분")
else:
    print("특징 파일이 모두 있어 건너뜁니다")

## 4. 증강과 특징 추출
### 4-1. 증강 함수 (클립 하나 → 32000샘플(2초) 창 하나)
0. 클립은 zip 에 든 그대로 읽습니다(`load_clip`). zip 에서 이미 무음을 자르고 2초 이하로 맞춰 두었으므로 2회차에서는 자르지 않습니다.
1. 음량 -6~+6dB (항상)
2. 잔향(RIR) 컨볼루션 50% — 말소리에 먼저 입히고, 원래 음량(RMS)으로 맞춤
3. 창의 **끝**에 붙이고 끝난 뒤 0~0.2초 여유, 앞쪽은 0
4. 배경 소음(AudioSet) SNR -5~15dB 75% (창 전체에 섞음)
5. 색잡음 SNR 10~30dB 25% (스펙트럼 기울기 -2~2: 보라·파랑·백색·분홍·갈색 사이)
6. int16 범위로 자르기

SNR 의 기준은 말소리 클립의 RMS 입니다.

In [ ]:
%%time
def load_clip(path):
    # zip 에서 이미 무음을 자르고 2초 이하로 맞춰 둔 클립을 그대로 읽음 (다시 자르지 않음)
    x, sr = sf.read(path, dtype="int16", always_2d=True)
    x = x.astype(np.float32).mean(axis=1)          # int16 크기 그대로의 float32
    return resample_to(x, sr, SR)


def scale_to_snr(noise, ref_rms, snr_db):
    n_rms = rms(noise)
    if n_rms < 1e-8:
        return np.zeros_like(noise)
    return (noise * (ref_rms / (n_rms * 10 ** (snr_db / 20)))).astype(np.float32)


def random_noise_segment(rng, n):
    src = noises[int(rng.integers(len(noises)))]
    if len(src) < n:
        src = np.tile(src, int(np.ceil(n / len(src))))
    s = int(rng.integers(0, len(src) - n + 1))
    return src[s:s + n]


def colored_noise(rng, n):
    beta = rng.uniform(-2.0, 2.0)                   # 0 백색, 1 분홍, 2 갈색, -1 파랑, -2 보라
    spec = np.fft.rfft(rng.standard_normal(n))
    f = np.fft.rfftfreq(n, d=1.0 / SR)
    f[0] = f[1]
    spec *= f ** (-beta / 2.0)
    return np.fft.irfft(spec, n).astype(np.float32)


def augment_clip(clip, rng):
    W = WINDOW_SAMPLES
    end_pad = int(rng.integers(0, int(END_PAD_MAX_S * SR) + 1))
    x = clip * np.float32(10 ** (rng.uniform(*GAIN_DB) / 20))       # 1) 음량
    if len(x) > W - end_pad:                                        # 2초 꽉 찬 클립: 여유를 줄이고 그래도 길면 앞을 자름
        end_pad = max(0, W - len(x))
        x = x[-(W - end_pad):]
    speech_rms = rms(x)
    start = W - end_pad - len(x)

    if rng.random() < RIR_P:                                        # 2) 잔향
        rir = rirs[int(rng.integers(len(rirs)))]
        wet = scipy.signal.fftconvolve(x, rir).astype(np.float32)
        wet *= speech_rms / max(rms(wet[:len(x)]), 1e-8)
        x = wet[:W - start]                                         # 잔향 꼬리는 창 끝에서 자름

    out = np.zeros(W, dtype=np.float32)                             # 3) 창의 끝에 배치
    out[start:start + len(x)] = x

    if rng.random() < BG_NOISE_P:                                   # 4) 배경 소음
        out += scale_to_snr(random_noise_segment(rng, W), speech_rms, rng.uniform(*BG_SNR_DB))
    if rng.random() < COLOR_NOISE_P:                                # 5) 색잡음
        out += scale_to_snr(colored_noise(rng, W), speech_rms, rng.uniform(*COLOR_SNR_DB))
    return np.clip(np.round(out), -32768, 32767).astype(np.int16)   # 6) int16 범위


def embed_in_chunks(fx, x, chunk=1024):
    outs = []
    for s in tqdm(range(0, len(x), chunk), desc="  임베딩", leave=False):
        outs.append(fx.embed_clips(x[s:s + chunk], batch_size=EMBED_BATCH, ncpu=EMBED_NCPU))
    e = np.concatenate(outs).astype(np.float32)
    assert e.shape[1] >= 16 and e.shape[2] == 96, e.shape
    return e[:, -16:, :]


print("증강 함수 준비 완료")

### 4-2. 증강 → 특징 (N, 16, 96) → Drive 에 .npy 저장
- positive_train 3번, negative_train 2번(돌 때마다 다르게 섞임), positive_test·negative_test 1번.
- 특징: `AudioFeatures(inference_framework="onnx", device=...).embed_clips(x_int16, batch_size=128, ncpu=4)`
- Drive 에 파일이 있으면 그 묶음은 건너뜁니다.

In [ ]:
%%time
feature_extractor = None
for i, split in enumerate(splits):
    out_path = feature_paths[split]
    if os.path.exists(out_path) and not FORCE_RECOMPUTE_FEATURES:
        print(f"{split}: 저장된 특징을 씁니다 {np.load(out_path, mmap_mode='r').shape}")
        continue
    assert len(noises) > 0 and len(rirs) > 0, "배경 소음/RIR 이 비어 있습니다. 3-4, 3-5 셀을 먼저 실행하세요."
    if feature_extractor is None:
        feature_extractor = AudioFeatures(inference_framework="onnx", device=feat_device)
        print("특징 추출 실행기:", feature_extractor.onnx_execution_provider)

    rng = np.random.default_rng([SEED, 10 + i])     # 묶음마다 고정된 난수열
    clips = [load_clip(p) for p in tqdm(wav_files[split], desc=f"{split} 읽기", leave=False)]
    secs = np.array([len(c) / SR for c in clips])
    print(f"{split}: 길이 평균 {secs.mean():.2f}초, 최소 {secs.min():.2f}, 최대 {secs.max():.2f} "
          f"| 2초 넘는 클립 {int((secs > WINDOW_SAMPLES / SR).sum())}개(끝 2초만 씀)")
    rounds = AUG_ROUNDS[split]
    x = np.empty((rounds * len(clips), WINDOW_SAMPLES), dtype=np.int16)
    k = 0
    with tqdm(total=len(x), desc=f"{split} 증강 ×{rounds}") as pbar:
        for r in range(rounds):
            for c in clips:
                x[k] = augment_clip(c, rng)
                k += 1
                pbar.update(1)

    feats_split = embed_in_chunks(feature_extractor, x)
    tmp_path = f"/content/{split}.npy"
    np.save(tmp_path, feats_split)
    shutil.copy(tmp_path, out_path)     # 다 쓴 뒤 Drive 로 복사 (중간에 끊겨도 깨진 파일이 남지 않게)
    os.remove(tmp_path)
    print(f"{split}: {feats_split.shape} → {out_path}")
    del x, clips, feats_split
    gc.collect()

## 5. 학습 준비
### 5-1. 특징 불러오기

In [ ]:
%%time
feats = {s: np.load(feature_paths[s]).astype(np.float32) for s in tqdm(splits, desc="특징 불러오기")}
for s in splits:
    assert feats[s].shape[1:] == (16, 96), (s, feats[s].shape)
    print(f"{s:15s} {feats[s].shape}")
print(f"배경 특징 {bg_feats.shape} | 오탐 검증 창 {n_val_windows:,}개")

### 5-2. 모델 (openWakeWord dnn 구조)
`Flatten → Linear(1536, L) → LayerNorm(L) → ReLU → [Linear(L, L) → LayerNorm(L) → ReLU] × N_BLOCKS → Linear(L, 1) → Sigmoid`
학습 때는 수치 안정성을 위해 Sigmoid 앞의 값(logit)으로 BCE 를 계산하고, 내보낼 때는 Sigmoid 까지 포함합니다.

In [ ]:
%%time
class JarviseoDNN(nn.Module):
    def __init__(self, layer_dim=LAYER_DIM, n_blocks=N_BLOCKS, input_shape=(16, 96)):
        super().__init__()
        layers = [nn.Flatten(),
                  nn.Linear(input_shape[0] * input_shape[1], layer_dim), nn.LayerNorm(layer_dim), nn.ReLU()]
        for _ in range(n_blocks):
            layers += [nn.Linear(layer_dim, layer_dim), nn.LayerNorm(layer_dim), nn.ReLU()]
        layers += [nn.Linear(layer_dim, 1)]
        self.body = nn.Sequential(*layers)
        self.sigmoid = nn.Sigmoid()

    def logits(self, x):
        return self.body(x)

    def forward(self, x):
        return self.sigmoid(self.body(x))


_m = JarviseoDNN()
print(_m)
print(f"파라미터 {sum(p.numel() for p in _m.parameters()):,}개")
del _m

### 5-3. 검증 함수
`TH_GRID`(0.5, 0.6, 0.7, 0.8, 0.9, 0.95)의 임계값마다 아래를 모두 기록합니다. 점수는 한 번만 계산하고 임계값별로 셉니다.
- (a) positive_test recall: 점수 ≥ t 비율
- (b) negative_test 오탐률: 점수 ≥ t 비율
- (c) 오탐 검증 특징의 시간당 오탐: 16프레임 창을 1프레임씩 밀며 점수를 내고, t 를 넘은 창을 셉니다. 한 번 넘으면 다음 25창(2초)은 세지 않습니다. 이 수 ÷ 10.7시간.

기록 이름은 `recall@0.8`, `neg_fpr@0.8`, `val_fp@0.8`, `fph@0.8` 처럼 붙고, 0.5 의 값은 2회차와 같은 이름(`recall`, `neg_fpr`, `val_fp`, `fp_per_hour`)으로도 남깁니다.

In [ ]:
%%time
assert SCORE_THRESHOLD in TH_GRID and TH_GRID == sorted(TH_GRID)
val_t = torch.from_numpy(val_feats).to(DEVICE)
pos_test_t = torch.from_numpy(feats["positive_test"]).to(DEVICE)
neg_test_t = torch.from_numpy(feats["negative_test"]).to(DEVICE)


def th_tag(t):
    return f"{t:g}"


@torch.no_grad()
def predict_batches(model, x, bs=8192):
    return torch.cat([model(x[s:s + bs]).squeeze(1) for s in range(0, len(x), bs)]).float().cpu().numpy()


@torch.no_grad()
def predict_val_windows(model, bs=32768):
    out = []
    for s in range(0, n_val_windows, bs):
        e = min(s + bs, n_val_windows)
        win = val_t[s:e + 15].unfold(0, 16, 1).transpose(1, 2)   # (e-s, 16, 96)
        out.append(model(win).squeeze(1))
    return torch.cat(out).float().cpu().numpy()


def count_activations(scores, threshold=SCORE_THRESHOLD, cooldown=FP_COOLDOWN_WINDOWS):
    count, next_ok = 0, 0
    for i in np.flatnonzero(scores >= threshold):
        if i >= next_ok:
            count += 1
            next_ok = i + cooldown + 1        # 다음 25창은 세지 않음
    return count


def evaluate(model):
    model.eval()
    p = predict_batches(model, pos_test_t)
    n = predict_batches(model, neg_test_t)
    v = predict_val_windows(model)            # 창 점수는 한 번만 계산
    out = {}
    for t in TH_GRID:
        fp = count_activations(v, threshold=t)
        k = th_tag(t)
        out[f"recall@{k}"] = float((p >= t).mean())
        out[f"neg_fpr@{k}"] = float((n >= t).mean())
        out[f"val_fp@{k}"] = int(fp)
        out[f"fph@{k}"] = fp / VAL_HOURS
    k = th_tag(SCORE_THRESHOLD)               # 2회차와 같은 이름 (임계값 0.5)
    out.update({"recall": out[f"recall@{k}"], "neg_fpr": out[f"neg_fpr@{k}"],
                "val_fp": out[f"val_fp@{k}"], "fp_per_hour": out[f"fph@{k}"]})
    return out


def t_star_of(rec):
    # 시간당 오탐 ≤ TARGET_FP_PER_HOUR 를 맞추는 가장 낮은 임계값 (없으면 None)
    for t in TH_GRID:
        if rec[f"fph@{th_tag(t)}"] <= TARGET_FP_PER_HOUR:
            return t
    return None


# 창 만들기 확인: unfold 결과가 직접 자른 창과 같은지
_w = val_t[:20].unfold(0, 16, 1).transpose(1, 2)
assert torch.equal(_w[3], val_t[3:19]) and _w.shape == (5, 16, 96)
# 쿨다운 확인: 0, 10, 26, 27 이 넘으면 → 0, 26 두 번
assert count_activations(np.array([1 if i in (0, 10, 26, 27) else 0 for i in range(40)], np.float32)) == 2
# t* 확인
assert t_star_of({f"fph@{th_tag(t)}": (1.0 if t < 0.8 else 0.3) for t in TH_GRID}) == 0.8
assert t_star_of({f"fph@{th_tag(t)}": 1.0 for t in TH_GRID}) is None

seed_everything(SEED)
_r = evaluate(JarviseoDNN().to(DEVICE))
print("학습 전 모델 (0.5):", {k: _r[k] for k in ["recall", "neg_fpr", "val_fp", "fp_per_hour"]}, "| t*:", t_star_of(_r))

## 6. 학습 (5회차: 학습률 워밍업·코사인 감소 + 가중치 이동 평균(EMA))
- 한 스텝 배치: positive 50 + 우리 negative 50 + 배경 특징 1024 (float16 → float32)
- positive 50개 중 `BATCH_POS_MIC`=10개는 positive_train 의 실제 녹음(source == "mic") 행에서, 나머지 40개는 positive_train 전체에서 뽑습니다.
  positive_train 특징의 k번째 행은 `wav_files["positive_train"][k % 클립 수]` 에서 나왔습니다(증강을 라운드 순서로 쌓았으므로).
- 손실: BCE. negative(라벨 0: 우리 negative + 배경) 가중치를 1 → `MAX_NEG_WEIGHT` 까지 스텝에 따라 선형으로 올립니다.
- **학습률(5회차)**: `LR_WARMUP_STEPS`(1000) 동안 0 → `LR`(1e-4) 선형으로 올리고, 그 뒤 `STEPS` 까지 `LR_MIN`(1e-6) 으로 코사인 감소. 매 스텝 `opt.param_groups` 의 lr 을 정합니다.
- **EMA(5회차)**: `EMA_DECAY`=0.999. 매 스텝 `opt.step()` 뒤 EMA 가중치를 갱신합니다.
  그래디언트는 원래 모델로 계산하고, **검증·snapshots·ONNX 내보내기·스트리밍 확인은 모두 EMA 가중치**로 합니다.
- Adam, `STEPS` 스텝. `EVAL_EVERY` 스텝마다 검증하고 그때의 EMA 가중치를 보관합니다.

In [ ]:
%%time
from torch.optim.swa_utils import AveragedModel

seed_everything(SEED)
gen = torch.Generator().manual_seed(SEED)

model = JarviseoDNN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
try:
    from torch.optim.swa_utils import get_ema_multi_avg_fn
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY))
except ImportError:  # 오래된 torch
    ema = AveragedModel(model, avg_fn=lambda avg, cur, n: EMA_DECAY * avg + (1 - EMA_DECAY) * cur)
ema_model = ema.module          # 검증·저장·내보내기에 쓰는 EMA 가중치


def lr_at(step):
    # step 은 1부터. 워밍업: 0 → LR 선형, 그 뒤 LR → LR_MIN 코사인
    if step <= LR_WARMUP_STEPS:
        return LR * step / max(1, LR_WARMUP_STEPS)
    progress = (step - LR_WARMUP_STEPS) / max(1, STEPS - LR_WARMUP_STEPS)
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1 + math.cos(math.pi * min(1.0, progress)))


assert abs(lr_at(LR_WARMUP_STEPS) - LR) < 1e-12 and abs(lr_at(STEPS) - LR_MIN) < 1e-12 and lr_at(1) > 0
print("학습률:", {s: f"{lr_at(s):.2e}" for s in [1, LR_WARMUP_STEPS // 2, LR_WARMUP_STEPS, STEPS // 4, STEPS // 2, 3 * STEPS // 4, STEPS]})

pos_tr = torch.from_numpy(feats["positive_train"]).to(DEVICE)
neg_tr = torch.from_numpy(feats["negative_train"]).to(DEVICE)
bg_t = torch.from_numpy(bg_feats)
if DEVICE.type == "cuda" and torch.cuda.mem_get_info()[0] > bg_feats.nbytes + 2e9:
    bg_t = bg_t.to(DEVICE)
print("배경 특징 위치:", bg_t.device)
n_pos, n_neg, n_bg = len(pos_tr), len(neg_tr), len(bg_t)

# positive_train 행별 source: k번째 행 ← wav_files["positive_train"][k % 클립 수]
n_pos_clips = len(wav_files["positive_train"])
assert n_pos == n_pos_clips * AUG_ROUNDS["positive_train"], (n_pos, n_pos_clips)
pos_row_src = wav_sources["positive_train"][np.arange(n_pos) % n_pos_clips]
pos_mic_idx = torch.from_numpy(np.flatnonzero(pos_row_src == "mic")).to(DEVICE)
n_mic_clips = int((wav_sources["positive_train"] == "mic").sum())
print(f"positive_train 행 {n_pos} | 실제 녹음(mic) 행 {len(pos_mic_idx)} "
      f"(클립 {n_mic_clips} × 증강 {AUG_ROUNDS['positive_train']}번)")
assert len(pos_mic_idx) == n_mic_clips * AUG_ROUNDS["positive_train"]
assert 0 <= BATCH_POS_MIC <= BATCH_POS and (BATCH_POS_MIC == 0 or len(pos_mic_idx) > 0)
n_pos_rest = BATCH_POS - BATCH_POS_MIC
print(f"스텝당 positive {BATCH_POS}개 = 실제 녹음에서 {BATCH_POS_MIC} + 전체에서 {n_pos_rest}")

y = torch.cat([torch.ones(BATCH_POS), torch.zeros(BATCH_NEG + BATCH_BG)]).to(DEVICE)
is_pos = y > 0.5

history, snapshots = [], {}
loss_acc, loss_n = torch.zeros((), device=DEVICE), 0
model.train()
pbar = tqdm(range(1, STEPS + 1), desc="학습", mininterval=1.0)
for step in pbar:
    lr_now = lr_at(step)
    for g in opt.param_groups:
        g["lr"] = lr_now
    neg_w = 1.0 + (MAX_NEG_WEIGHT - 1.0) * (step - 1) / max(1, STEPS - 1)
    ip_mic = pos_mic_idx[torch.randint(0, max(1, len(pos_mic_idx)), (BATCH_POS_MIC,), generator=gen).to(DEVICE)]
    ip_all = torch.randint(0, n_pos, (n_pos_rest,), generator=gen).to(DEVICE)
    ineg = torch.randint(0, n_neg, (BATCH_NEG,), generator=gen)
    ib = torch.randint(0, n_bg, (BATCH_BG,), generator=gen)
    xb = bg_t[ib.to(bg_t.device)].to(DEVICE, non_blocking=True).float()
    x = torch.cat([pos_tr[ip_mic], pos_tr[ip_all], neg_tr[ineg.to(DEVICE)], xb])
    w = torch.where(is_pos, torch.ones_like(y), torch.full_like(y, neg_w))

    loss = F.binary_cross_entropy_with_logits(model.logits(x).squeeze(1), y, weight=w)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    ema.update_parameters(model)          # EMA 갱신 (opt.step() 뒤)
    loss_acc += loss.detach()
    loss_n += 1

    if step % EVAL_EVERY == 0 or step == STEPS:
        m = evaluate(ema_model)            # 검증은 EMA 가중치로
        history.append({"step": step, "lr": lr_now, "neg_weight": round(neg_w, 2),
                        "train_loss": float(loss_acc.item() / loss_n), **m})
        snapshots[step] = {k: v.detach().cpu().clone() for k, v in ema_model.state_dict().items()}
        loss_acc, loss_n = torch.zeros((), device=DEVICE), 0
        ts = t_star_of(m)
        pbar.set_postfix(lr=f"{lr_now:.1e}", recall05=f"{m['recall']:.3f}", fph05=f"{m['fp_per_hour']:.2f}",
                         t_star=("-" if ts is None else ts),
                         recall_ts=("-" if ts is None else f"{m[f'recall@{th_tag(ts)}']:.3f}"))
        model.train()
print(f"검증 {len(history)}회 기록 (EMA 가중치)")

## 7. 검증 기록과 모델 선택 (3회차에서 바뀜)
- 후보는 `STEPS × SELECT_FROM_FRAC` = 10000 스텝부터의 검증 시점만 씁니다(2회차는 학습 초반 1500 스텝이 뽑혔음).
- 스텝마다 시간당 오탐 ≤ `TARGET_FP_PER_HOUR`(0.5)를 맞추는 가장 낮은 임계값 **t*** 를 `TH_GRID` 에서 찾고, **t* 에서의 recall** 로 스텝을 비교합니다.
  같으면 t* 에서의 시간당 오탐이 낮은 쪽, 그다음은 늦은 스텝을 고릅니다.
- 어떤 후보도 0.95 에서 시간당 오탐 ≤ 0.5 를 못 맞추면, 0.95 에서 시간당 오탐이 가장 낮은 스텝을 고르고 경고를 띄웁니다.
- **t* 는 참고값입니다.** 실제 임계값은 학습 뒤 eval 로 정합니다.
- 그래프: 위 recall(0.5 와 t*), 가운데 시간당 오탐(0.5 와 t*), 아래 t*. 그래프 글꼴 문제를 피하려고 라벨은 영어로 씁니다.

In [ ]:
%%time
hist_df = pd.DataFrame(history)
t_stars = [t_star_of(r) for r in history]
hist_df["t_star"] = [np.nan if t is None else t for t in t_stars]
for name in ["recall", "neg_fpr", "val_fp", "fph"]:
    hist_df[f"{name}_t*"] = [np.nan if t is None else r[f"{name}@{th_tag(t)}"] for r, t in zip(history, t_stars)]

select_from_step = int(round(STEPS * SELECT_FROM_FRAC))
cand = hist_df[hist_df.step >= select_from_step]
ok = cand[cand.t_star.notna()]
target_met = len(ok) > 0
if target_met:
    best_row = ok.sort_values(["recall_t*", "fph_t*", "step"], ascending=[False, True, False]).iloc[0]
    best_t = float(best_row.t_star)
else:
    best_t = TH_GRID[-1]
    k = th_tag(best_t)
    print(f"※ 경고: {select_from_step} 스텝 이후 어떤 후보도 임계값 {best_t} 에서 시간당 오탐 ≤ {TARGET_FP_PER_HOUR} 를 맞추지 못했습니다. "
          f"{best_t} 에서 시간당 오탐이 가장 낮은 스텝을 고릅니다.")
    best_row = cand.sort_values([f"fph@{k}", f"recall@{k}", "step"], ascending=[True, False, False]).iloc[0]
best_step = int(best_row.step)
bk = th_tag(best_t)
best_rec = history[[r["step"] for r in history].index(best_step)]
best_metrics = {"threshold": best_t, "recall": best_rec[f"recall@{bk}"], "neg_fpr": best_rec[f"neg_fpr@{bk}"],
                "val_fp": best_rec[f"val_fp@{bk}"], "fp_per_hour": best_rec[f"fph@{bk}"]}

show_cols = ["step", "neg_weight", "train_loss", "recall", "fp_per_hour", "t_star", "recall_t*", "neg_fpr_t*", "val_fp_t*", "fph_t*"]
with pd.option_context("display.max_rows", None, "display.width", 220):
    def _f(fmt):
        return lambda v: "-" if pd.isna(v) else format(v, fmt)
    print(hist_df[show_cols].to_string(index=False, formatters={
        "train_loss": _f(".4f"), "recall": _f(".4f"), "fp_per_hour": _f(".3f"),
        "t_star": _f("g"), "recall_t*": _f(".4f"), "neg_fpr_t*": _f(".4f"),
        "val_fp_t*": lambda v: "-" if pd.isna(v) else f"{int(v)}", "fph_t*": _f(".3f")}))
print("(recall·fp_per_hour 는 임계값 0.5, *_t* 는 t* 에서의 값. 임계값별 전체 값은 JSON 의 validation_history 에 있습니다.)")
print(f"\n후보: {select_from_step} 스텝부터 {len(cand)}개 | t* 를 찾은 후보 {len(ok)}개")
print(f"고른 스텝: {best_step} | t* = {best_t:g}" + ("" if target_met else " (목표 미달, 0.95 기준)") +
      f" | recall {best_metrics['recall']:.4f} | negative 오탐률 {best_metrics['neg_fpr']:.4f}"
      f" | 시간당 오탐 {best_metrics['fp_per_hour']:.3f} ({int(best_metrics['val_fp'])}회 / {VAL_HOURS}시간)")
print(f"참고: 같은 스텝의 임계값 0.5 값 → recall {best_rec['recall']:.4f}, 시간당 오탐 {best_rec['fp_per_hour']:.3f}")
print("※ t* 는 참고값입니다. 실제 임계값은 학습 뒤 eval 로 정합니다.")

# 학습이 안정됐는지 보기: 4000 스텝마다 recall@0.5, fph@0.5, fph@0.9 (EMA 가중치)
stab = hist_df[hist_df.step % 4000 == 0][["step", "lr", "recall@0.5", "fph@0.5", "fph@0.9"]]
print("\n4000 스텝마다 (EMA):")
print(stab.to_string(index=False, formatters={"lr": "{:.1e}".format, "recall@0.5": "{:.4f}".format,
                                              "fph@0.5": "{:.3f}".format, "fph@0.9": "{:.3f}".format}))
stability_rows = stab.to_dict("records")

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
axes[0].plot(hist_df.step, hist_df.recall, label="positive_test recall @0.5", alpha=0.6)
axes[0].plot(hist_df.step, hist_df["recall_t*"], "o-", ms=3, label="positive_test recall @t*")
axes[0].plot(hist_df.step, hist_df.neg_fpr, label="negative_test false-positive rate @0.5", alpha=0.6)
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_ylabel("rate")
axes[0].legend(loc="lower right", fontsize=8)
axes[1].plot(hist_df.step, hist_df.fp_per_hour, color="tab:red", alpha=0.5, label="false activations / hour @0.5")
axes[1].plot(hist_df.step, hist_df["fph_t*"], "o-", ms=3, color="tab:purple", label="false activations / hour @t*")
axes[1].axhline(TARGET_FP_PER_HOUR, color="gray", ls="--", label=f"target {TARGET_FP_PER_HOUR}/h")
axes[1].set_yscale("symlog", linthresh=0.1)
axes[1].set_ylabel("FP / hour (symlog)")
axes[1].legend(loc="upper left", fontsize=8)
axes[2].step(hist_df.step, hist_df.t_star, where="mid", color="tab:green", label="t* (lowest threshold with FP/h <= target)")
axes[2].set_ylim(min(TH_GRID) - 0.05, 1.0)
axes[2].set_ylabel("t*")
axes[2].set_xlabel("step")
axes[2].legend(loc="lower right", fontsize=8)
for ax in axes:
    ax.axvline(select_from_step, color="gray", ls=":", lw=1.2)
    ax.axvline(best_step, color="green", ls=":", lw=1.5)
    ax.grid(alpha=0.3)
axes[0].set_title(f"jarviseo {RUN_TAG} validation (candidates from step {select_from_step}, selected step {best_step}, t*={best_t:g})")
plt.tight_layout()
plt.show()

## 8. ONNX 내보내기
`torch.onnx.export`, opset 13, 입력 이름 `"input"`, 모양 `(1, 16, 96)` float32, 0번 축만 dynamic, 출력 `(1, 1)`.
openwakeword.Model 은 `get_inputs()[0].shape[1]` 을 16, `get_outputs()[0].shape[1]` 을 1 로 읽으므로 이 두 축은 고정 숫자여야 합니다.
torch 2.9 부터 기본값이 dynamo 내보내기라서 opset 13 을 쓰려고 `dynamo=False`(TorchScript 내보내기)를 명시합니다.

In [ ]:
%%time
export_model = JarviseoDNN()
export_model.load_state_dict(snapshots[best_step])
export_model.eval()

local_onnx = f"/content/{MODEL_NAME}.onnx"
dummy = torch.zeros(1, 16, 96, dtype=torch.float32)
exporter = "torchscript"
try:
    torch.onnx.export(export_model, (dummy,), local_onnx, dynamo=False, opset_version=ONNX_OPSET,
                      input_names=["input"], output_names=["output"],
                      dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}})
except Exception as err:
    # TorchScript 내보내기가 없는 torch 일 때만: dynamo 내보내기 (opset 18 이상만 지원)
    print("TorchScript 내보내기 실패 → dynamo 로 재시도:", repr(err))
    exporter = "dynamo"
    torch.onnx.export(export_model, (torch.zeros(2, 16, 96),), local_onnx, dynamo=True,
                      opset_version=max(ONNX_OPSET, 18), input_names=["input"], output_names=["output"],
                      dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=65536)}},
                      external_data=False)

sess = ort.InferenceSession(local_onnx, providers=["CPUExecutionProvider"])
inp, outp = sess.get_inputs()[0], sess.get_outputs()[0]
print("입력:", inp.name, inp.shape, inp.type, "| 출력:", outp.name, outp.shape, outp.type)
assert inp.name == "input" and inp.type == "tensor(float)"
assert list(inp.shape[1:]) == [16, 96], inp.shape
assert outp.shape[1] == 1, outp.shape
assert not isinstance(inp.shape[0], int), "0번 축이 dynamic 이 아닙니다"

try:
    import onnx
    onnx_opset = max(o.version for o in onnx.load(local_onnx).opset_import if o.domain in ("", "ai.onnx"))
except Exception:
    onnx_opset = None
print("내보내기:", exporter, "| opset:", onnx_opset)
if onnx_opset is not None and onnx_opset != ONNX_OPSET:
    print(f"※ opset 이 {ONNX_OPSET} 이 아닙니다 ({onnx_opset}). onnxruntime 에서는 동작합니다.")

xt = feats["positive_test"][:64]
with torch.no_grad():
    ref = export_model(torch.from_numpy(xt)).numpy()
got = sess.run(None, {"input": xt})[0]
print("배치 64 출력 모양:", got.shape, "| torch 와 최대 차이:", float(np.abs(got - ref).max()))
assert got.shape == (64, 1) and np.abs(got - ref).max() < 1e-4
assert sess.run(None, {"input": xt[:1]})[0].shape == (1, 1)

shutil.copy(local_onnx, ONNX_PATH)
print(f"저장: {ONNX_PATH} ({os.path.getsize(ONNX_PATH) / 1024:.1f} KB)")

## 9. openwakeword.Model 로 스트리밍 확인
`openwakeword.model.Model(wakeword_models=[onnx 경로], inference_framework="onnx")` 로 엽니다.
zip 의 positive_test(384)·negative_test(291) wav(증강 없음) 앞뒤에 무음 1초를 붙이고 1280샘플(80ms)씩 `predict` 에 흘려 파일별 최고 점수를 남깁니다.
- 2회차와 같은 표: 임계값 0.1~0.9 마다 FRR(positive 중 못 넘은 비율)·FAR(negative 중 넘은 비율), 전체와 source(합성/실제)별
- 3회차에 더한 표: positive_test 를 말투(정상·끊음·늘임)별로 나눈 FRR, 임계값 0.5 와 t* 에서
- 1·2회차와 나란히 놓는 비교표는 뺐습니다(test 구성이 다릅니다).

In [ ]:
%%time
seed_everything(SEED)
oww = OWWModel(wakeword_models=[local_onnx], inference_framework="onnx")
oww_key = os.path.splitext(os.path.basename(local_onnx))[0]
print("불러온 모델:", list(oww.models.keys()), "| 입력 프레임:", oww.model_inputs, "| 출력:", oww.model_outputs)
assert oww.model_inputs[oww_key] == 16 and oww.model_outputs[oww_key] == 1

CHUNK = 1280
silence = np.zeros(SR, dtype=np.int16)


def stream_max_score(path):
    x, sr = sf.read(path, dtype="int16", always_2d=True)
    x = x[:, 0] if x.shape[1] == 1 else x.mean(axis=1).astype(np.int16)
    if sr != SR:
        x = np.clip(np.round(resample_to(x.astype(np.float32), sr, SR)), -32768, 32767).astype(np.int16)
    x = np.concatenate([silence, x, silence])
    x = np.concatenate([x, np.zeros((-len(x)) % CHUNK, dtype=np.int16)])
    oww.reset()
    best = 0.0
    for s in range(0, len(x), CHUNK):
        best = max(best, float(oww.predict(x[s:s + CHUNK])[oww_key]))
    return best


stream_scores = {s: np.array([stream_max_score(p) for p in tqdm(wav_files[s], desc=f"{s} 스트리밍")])
                 for s in ["positive_test", "negative_test"]}
pos_s, neg_s = stream_scores["positive_test"], stream_scores["negative_test"]

check_rows = []
for t in CHECK_THRESHOLDS:
    check_rows.append({"threshold": t,
                       "FRR": float((pos_s < t).mean()), "missed": int((pos_s < t).sum()),
                       "FAR": float((neg_s >= t).mean()), "false_accepts": int((neg_s >= t).sum())})
check_df = pd.DataFrame(check_rows)
print(f"positive_test {len(pos_s)}개, negative_test {len(neg_s)}개 (파일별 최고 점수 기준)")
print(check_df.to_string(index=False, formatters={"threshold": "{:.1f}".format,
                                                  "FRR": "{:.4f}".format, "FAR": "{:.4f}".format}))

# 실제 녹음(mic)과 합성(tts)을 따로
pos_src = wav_sources["positive_test"]
neg_src = wav_sources["negative_test"]
src_rows = []
for src in sorted(set(pos_src) | set(neg_src)):
    p, n = pos_s[pos_src == src], neg_s[neg_src == src]
    for t in CHECK_THRESHOLDS:
        src_rows.append({"source": src, "threshold": t,
                         "FRR": float((p < t).mean()) if len(p) else float("nan"), "n_pos": len(p),
                         "FAR": float((n >= t).mean()) if len(n) else float("nan"), "n_neg": len(n)})
src_df = pd.DataFrame(src_rows)
print("\nsource 별 (mic = 실제 녹음, tts = 합성):")
print(src_df.to_string(index=False, formatters={"threshold": "{:.1f}".format,
                                                "FRR": "{:.4f}".format, "FAR": "{:.4f}".format}))


# positive_test 말투별 FRR (임계값 0.5 와 t*)
def fmt_rate(m, n):
    return f"{m / n:.1%} ({m}/{n})" if n else "-"


pos_style = wav_styles["positive_test"]
style_rows = []
for style in ["정상", "끊음", "늘임", "전체"]:
    mask = np.ones(len(pos_s), bool) if style == "전체" else (pos_style == style)
    p = pos_s[mask]
    row = {"말투": style, "n": int(mask.sum()),
           "실제(mic) 수": int((pos_src[mask] == "mic").sum()), "합성(tts) 수": int((pos_src[mask] == "tts").sum())}
    for t in [SCORE_THRESHOLD, best_t]:
        row[f"FRR@{t:g}"] = fmt_rate(int((p < t).sum()), len(p))
    style_rows.append(row)
style_df = pd.DataFrame(style_rows)
print(f"\npositive_test 말투별 FRR (임계값 {SCORE_THRESHOLD:g} 와 t* = {best_t:g}):")
print(style_df.to_string(index=False))

# negative_test label 별 FAR (임계값 0.5, 0.9, t*)
neg_label = wav_labels["negative_test"]
label_rows = []
for lab in sorted(set(neg_label)) + ["전체"]:
    mask = np.ones(len(neg_s), bool) if lab == "전체" else (neg_label == lab)
    n = neg_s[mask]
    row = {"label": lab, "n": int(mask.sum()),
           "실제(mic) 수": int((neg_src[mask] == "mic").sum()), "합성(tts) 수": int((neg_src[mask] == "tts").sum())}
    for t in sorted({SCORE_THRESHOLD, 0.9, best_t}):
        row[f"FAR@{t:g}"] = fmt_rate(int((n >= t).sum()), len(n))
    label_rows.append(row)
label_df = pd.DataFrame(label_rows)
print(f"\nnegative_test label 별 FAR (임계값 {SCORE_THRESHOLD:g}, 0.9, t* = {best_t:g}):")
print(label_df.to_string(index=False))

## 10. 학습 기록 저장 (`jarviseo-r6-train.json`)
설정값, 고른 스텝·t*·그때의 검증 수치, 전체 검증 기록(임계값별), 스트리밍 확인 표(전체·source별·말투별), 버전 정보를 Drive 에 저장합니다.

In [ ]:
%%time
def to_jsonable(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)


train_info = {
    "model": MODEL_NAME,
    "run_tag": RUN_TAG,
    "onnx_file": os.path.basename(ONNX_PATH),
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "config": CONFIG,
    "selection": {
        "rule": (f"step >= {select_from_step} (STEPS x {SELECT_FROM_FRAC}) 인 검증 시점 중, "
                 f"fph <= {TARGET_FP_PER_HOUR} 를 맞추는 가장 낮은 임계값 t* (TH_GRID {TH_GRID}) 에서의 recall 최대. "
                 f"같으면 fph 낮은 쪽, 그다음 늦은 스텝. 없으면 {TH_GRID[-1]} 에서 fph 최소 스텝 (경고)."),
        "target_met": bool(target_met),
        "selected_step": best_step,
        "t_star": best_t,
        "t_star_note": "t* 는 참고값이고 실제 임계값은 학습 뒤 eval 로 정한다",
        "weights": f"EMA (decay {EMA_DECAY}), lr warmup {LR_WARMUP_STEPS} + cosine to {LR_MIN}",
        "stability_every_4000": stability_rows,
        "metrics_at_t_star": best_metrics,
        "metrics_at_0.5": {k: best_rec[k] for k in ["recall", "neg_fpr", "val_fp", "fp_per_hour"]},
        "candidates": int(len(cand)), "candidates_with_t_star": int(len(ok)),
    },
    "validation_history": [{**r, "t_star": t} for r, t in zip(history, t_stars)],
    "stream_check": {"positive_test_files": int(len(pos_s)), "negative_test_files": int(len(neg_s)),
                     "silence_pad_s": 1.0, "chunk_samples": CHUNK, "table": check_rows,
                     "by_source": src_rows, "positive_by_style": style_rows, "negative_by_label": label_rows},
    "data": {
        "zip_path": ZIP_PATH,
        "data_root": data_root,
        "wav_counts": {s: len(wav_files[s]) for s in splits},
        "split_by_source": {s: pd.Series(wav_sources[s]).value_counts().to_dict() for s in splits},
        "positive_by_style": {s: pd.Series(wav_styles[s]).value_counts().to_dict() for s in ["positive_train", "positive_test"]},
        "positive_train_mic_rows": int(len(pos_mic_idx)),
        "feature_shapes": {s: list(feats[s].shape) for s in splits},
        "background_rows_in_ram": int(bg_feats.shape[0]),
        "validation_windows": int(n_val_windows),
        "rir_count": len(rirs),
        "audioset": audioset_stats,
    },
    "onnx": {"exporter": exporter, "opset": onnx_opset,
             "input": {"name": inp.name, "shape": [str(d) for d in inp.shape], "type": inp.type},
             "output": {"name": outp.name, "shape": [str(d) for d in outp.shape], "type": outp.type}},
    "runtime": {"train_device": str(DEVICE),
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "feature_device": feat_device, "torch": torch.__version__, "onnxruntime": ort.__version__,
                "numpy": np.__version__, "openwakeword": importlib.metadata.version("openwakeword")},
}
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(train_info, f, ensure_ascii=False, indent=2, default=to_jsonable)

print("저장:", JSON_PATH)
print("저장:", ONNX_PATH)
print(f"고른 스텝 {best_step} | t* {best_t:g} | recall@t* {best_metrics['recall']:.4f} "
      f"| 시간당 오탐@t* {best_metrics['fp_per_hour']:.3f} | 목표 충족 {target_met}")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    if os.path.isfile(p):
        print(f"  {os.path.basename(p):30s} {os.path.getsize(p) / 1024:10.1f} KB")